<a href="https://colab.research.google.com/github/fotomain/py_labs_ai_crash_course/blob/main/CAPSTONE_V2_OR_TOOLS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "pandas==2.2.2" "numpy<2.1,>=1.22" "protobuf>=5.26.1,<6.0.0dev" "ortools==9.12.4544"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [2]:
import pandas as pd
import numpy as np
from prophet import Prophet
import gdown
import os
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 0. Download data from Google Drive
# ------------------------------------------------------------
def download_data_from_drive():
    file_id = '1lWe3IB8L_3TpDHO1MRKDqaH1Tk20pzDk'
    url = f'https://drive.google.com/uc?id={file_id}'
    output = 'sales_data.csv'
    if not os.path.exists(output):
        print('[0] Downloading sales_data.csv...')
        gdown.download(url, output, quiet=False)
    else:
        print('[0] File already exists.')
    return output

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
def load_data(filepath):
    data = pd.read_csv(filepath)
    data['Month'] = pd.to_datetime(data['Month'])
    data.set_index('Month', inplace=True)
    return data

# ------------------------------------------------------------
# 2. Forecast with Prophet
# ------------------------------------------------------------
def forecast_sales(data, periods=24):
    df = data.rename(columns={'#Passengers': 'y'})
    df['ds'] = df.index
    df = df[['ds', 'y']]
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    model.fit(df)
    future = model.make_future_dataframe(periods=periods, freq='MS')
    forecast = model.predict(future)
    forecast_1961 = forecast[forecast['ds'].dt.year == 1961]
    predicted = pd.DataFrame({
        'Month': forecast_1961['ds'].dt.strftime('%Y-%m'),
        'Predicted_Passengers': np.round(forecast_1961['yhat']).astype(int)
    })
    predicted['Month'] = pd.to_datetime(predicted['Month'])
    predicted.set_index('Month', inplace=True)
    return predicted, model

# ------------------------------------------------------------
# 3. MILP Optimizer (using OR-Tools)
# ------------------------------------------------------------
def optimize_orders_milp(predicted_plan, max_order=100, min_order=0):
    """
    Solve MILP to minimize the total number of orders.
    Constraints:
        - Weekly demand = monthly demand / 4
        - Inventory balance: I_t = I_{t-1} + O_t - d_t
        - No stockouts: I_t >= 0
        - Order capacity: 0 <= O_t <= max_order
        - If O_t > 0 then O_t >= min_order (if min_order > 0)
    Returns DataFrame with columns: week, demand, order_qty, inventory_after
    """
    # Convert monthly demand to weekly
    weekly_demands = []
    for month, row in predicted_plan.iterrows():
        monthly_demand = row['Predicted_Passengers']
        weekly = monthly_demand / 4.0
        weekly_demands.extend([weekly] * 4)
    weeks = len(weekly_demands)

    # Try to get a solver
    try:
        from ortools.linear_solver import pywraplp
    except ImportError:
        return None  # OR-Tools not installed

    solver = pywraplp.Solver.CreateSolver('CBC')
    if not solver:
        solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver:
        return None

    # Decision variables
    O = [solver.IntVar(0, max_order, f'O_{t}') for t in range(weeks)]
    I = [solver.NumVar(0, solver.infinity(), f'I_{t}') for t in range(weeks)]
    Y = [solver.IntVar(0, 1, f'Y_{t}') for t in range(weeks)]

    # Constraints
    for t in range(weeks):
        if t == 0:
            solver.Add(I[t] == O[t] - weekly_demands[t])
        else:
            solver.Add(I[t] == I[t-1] + O[t] - weekly_demands[t])
        solver.Add(I[t] >= 0)

    # Link O_t to Y_t
    for t in range(weeks):
        solver.Add(O[t] <= max_order * Y[t])
        if min_order > 0:
            solver.Add(O[t] >= min_order * Y[t])

    # Objective: minimize total orders
    objective = solver.Objective()
    for t in range(weeks):
        objective.SetCoefficient(Y[t], 1)
    objective.SetMinimization()

    status = solver.Solve()
    if status != pywraplp.Solver.OPTIMAL:
        return None  # infeasible or other error

    # Build result DataFrame
    order_df = pd.DataFrame({
        'week': range(1, weeks+1),
        'demand': [round(d, 1) for d in weekly_demands],
        'order_qty': [int(O[t].solution_value()) for t in range(weeks)],
        'inventory_after': [round(I[t].solution_value(), 1) for t in range(weeks)]
    })
    return order_df

# ------------------------------------------------------------
# 4. Heuristic (fallback)
# ------------------------------------------------------------
def generate_orders_heuristic(predicted_plan, max_order=100, safety_stock_factor=0.2):
    orders = []
    stock = 0.0
    week_counter = 0
    for month, row in predicted_plan.iterrows():
        monthly_demand = row['Predicted_Passengers']
        weekly_demand = monthly_demand / 4.0
        safety_stock = weekly_demand * safety_stock_factor
        for _ in range(4):
            week_counter += 1
            needed = max(0.0, weekly_demand + safety_stock - stock)
            order_qty = min(max_order, needed) if needed > 0 else 0
            stock += order_qty
            stock -= weekly_demand
            stock = max(0.0, stock)
            orders.append({
                'week': week_counter,
                'order_qty': int(order_qty),
                'demand': round(weekly_demand, 1),
                'inventory_after': round(stock, 1)
            })
    return pd.DataFrame(orders)

# ------------------------------------------------------------
# 5. Save outputs
# ------------------------------------------------------------
def save_outputs(predicted_plan, order_df):
    predicted_plan.to_csv('predicted_plan.csv')
    print('✓ Saved predicted_plan.csv')

    order_df['month_num'] = (order_df['week'] - 1) // 4 + 1
    for m in range(1, 13):
        month_data = order_df[order_df['month_num'] == m]
        month_data[['week', 'order_qty', 'demand', 'inventory_after']].to_csv(
            f'order_month_{m:02d}.csv', index=False
        )
        print(f'✓ Saved order_month_{m:02d}.csv')

# ------------------------------------------------------------
# 6. Main
# ------------------------------------------------------------
def main():
    print('=' * 70)
    print(' SALES FORECASTING + OPTIMAL ORDER PLANNING (MILP)')
    print('=' * 70)

    csv_file = download_data_from_drive()
    print('\n[1] Loading sales data...')
    data = load_data(csv_file)
    print(f'    Loaded {len(data)} monthly records.')

    print('\n[2] Forecasting sales for 1961...')
    predicted_plan, _ = forecast_sales(data)
    print(f'    Forecast generated for {len(predicted_plan)} months.')

    print('\n[3] Solving optimal order schedule (MILP)...')
    order_df = optimize_orders_milp(predicted_plan)
    if order_df is not None:
        print('    MILP succeeded!')
        method = 'MILP'
    else:
        print('    MILP failed or OR-Tools unavailable. Falling back to heuristic.')
        order_df = generate_orders_heuristic(predicted_plan)
        method = 'Heuristic'

    print(f'[4] Generated {len(order_df)} weekly orders using {method}.')
    save_outputs(predicted_plan, order_df)

    total_orders = order_df['order_qty'].gt(0).sum()
    total_units = int(order_df['order_qty'].sum())
    avg_order = order_df['order_qty'].mean()

    print('\n' + '=' * 70)
    print(' SUMMARY STATISTICS')
    print('=' * 70)
    print(f'Total Annual Demand            : {int(predicted_plan["Predicted_Passengers"].sum()):,} units')
    print(f'Total Orders Placed            : {total_orders}')
    print(f'Total Units Ordered            : {total_units:,} units')
    print(f'Average Order Quantity         : {avg_order:.1f} units')
    print(f'Maximum Order Quantity         : {int(order_df["order_qty"].max())} units')
    print(f'Method used                    : {method}')

    print('\nPeak Demand Months (Top 3):')
    peak = predicted_plan.nlargest(3, 'Predicted_Passengers')
    for month, row in peak.iterrows():
        print(f'    {month.strftime("%B %Y")}: {int(row["Predicted_Passengers"])} passengers')

    print('\n' + '=' * 70)
    print('✅ Process completed successfully!')
    print('=' * 70)

if __name__ == '__main__':
    main()

 SALES FORECASTING + OPTIMAL ORDER PLANNING (MILP)
[0] Downloading sales_data.csv...


Downloading...
From: https://drive.google.com/uc?id=1lWe3IB8L_3TpDHO1MRKDqaH1Tk20pzDk
To: /content/sales_data.csv
100%|██████████| 1.75k/1.75k [00:00<00:00, 3.49MB/s]



[1] Loading sales data...
    Loaded 144 monthly records.

[2] Forecasting sales for 1961...


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


    Forecast generated for 12 months.

[3] Solving optimal order schedule (MILP)...
    MILP failed or OR-Tools unavailable. Falling back to heuristic.
[4] Generated 48 weekly orders using Heuristic.
✓ Saved predicted_plan.csv
✓ Saved order_month_01.csv
✓ Saved order_month_02.csv
✓ Saved order_month_03.csv
✓ Saved order_month_04.csv
✓ Saved order_month_05.csv
✓ Saved order_month_06.csv
✓ Saved order_month_07.csv
✓ Saved order_month_08.csv
✓ Saved order_month_09.csv
✓ Saved order_month_10.csv
✓ Saved order_month_11.csv
✓ Saved order_month_12.csv

 SUMMARY STATISTICS
Total Annual Demand            : 6,074 units
Total Orders Placed            : 48
Total Units Ordered            : 4,800 units
Average Order Quantity         : 100.0 units
Maximum Order Quantity         : 100 units
Method used                    : Heuristic

Peak Demand Months (Top 3):
    August 1961: 578 passengers
    July 1961: 577 passengers
    June 1961: 538 passengers

✅ Process completed successfully!
